<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/02_ML_Engineer/intermediate/06_feature_stores.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Feature Stores with Feast

> **Track:** ML Engineer | **Level:** Intermediate | **Time:** 2-3 hours

## Overview
A **feature store** is a centralized system for storing, serving, and sharing ML features. Without one, every team recomputes the same features, training-serving skew causes silent failures, and point-in-time correct joins for time-series data are error-prone.

### What You'll Learn
- The feature store problem: training-serving skew
- Feast architecture: registry, offline store, online store
- Defining feature views and entities
- Materializing features to the online store
- Point-in-time correct joins for training data
- Serving features in real-time

```bash
pip install feast pandas numpy scikit-learn
```

In [ ]:
# Setup: create a sample dataset simulating a ride-sharing platform
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from pathlib import Path

np.random.seed(42)

# Simulate driver feature data (historical)
n_drivers = 200
driver_ids = list(range(1001, 1001 + n_drivers))

driver_features = pd.DataFrame({
    "driver_id": driver_ids,
    "event_timestamp": [datetime(2024, 1, 1) + timedelta(hours=i % 24) for i in range(n_drivers)],
    "trips_completed": np.random.randint(50, 500, n_drivers),
    "avg_rating": np.round(np.random.uniform(3.5, 5.0, n_drivers), 2),
    "acceptance_rate": np.round(np.random.uniform(0.7, 1.0, n_drivers), 3),
    "conv_rate": np.round(np.random.uniform(0.6, 0.9, n_drivers), 3),
    "created": datetime(2024, 1, 1)
})

# Save to parquet (Feast uses parquet as offline store in local mode)
data_dir = Path("feature_store_data")
data_dir.mkdir(exist_ok=True)
driver_features.to_parquet(data_dir / "driver_features.parquet", index=False)

print(f"Created driver feature dataset: {driver_features.shape}")
print(driver_features.head(3))

## 1. The Training-Serving Skew Problem

**Training-serving skew** happens when the features used during training differ from those used at inference time — one of the most common silent failure modes in production ML.

In [ ]:
# Demonstrate training-serving skew (the problem feature stores solve)

print("=== THE TRAINING-SERVING SKEW PROBLEM ===")
print()
print("TRAINING: compute avg_rating from full historical data")
avg_rating_train = driver_features["avg_rating"].mean()
print(f"  Training avg_rating mean: {avg_rating_train:.3f}")

print()
print("SERVING: accidentally include newer/future data or use different window")
# Simulating a bug: wrong time window used in serving
avg_rating_serve = driver_features["avg_rating"].mean() + 0.2  # artificially inflated
print(f"  Serving avg_rating mean: {avg_rating_serve:.3f}")

print()
print(f"Skew magnitude: {abs(avg_rating_train - avg_rating_serve):.3f}")
print("This causes model predictions to degrade silently in production!")
print()
print("SOLUTION: Feature store ensures identical feature computation for train + serve")

## 2. Feast Project Setup

A Feast project consists of:
- **Entity**: the object features describe (driver, customer, product)
- **Feature View**: a group of features computed from a data source
- **Feature Service**: a curated set of features for a specific model

In [ ]:
# Define Feast feature store configuration (programmatic approach)
# In production: this goes in feature_store.yaml and separate Python files

# Check if feast is installed
try:
    import feast
    from feast import Entity, FeatureView, FileSource, Field, FeatureStore
    from feast.types import Float32, Int64
    FEAST_AVAILABLE = True
    print(f"Feast version: {feast.__version__}")
except ImportError:
    FEAST_AVAILABLE = False
    print("Feast not installed. Showing conceptual code below.")
    print("Install with: pip install feast")

FEATURE_STORE_YAML = """
# feature_store.yaml
project: driver_ranking
registry: data/registry.db
provider: local
online_store:
  type: sqlite
  path: data/online_store.db
offline_store:
  type: file
entity_key_serialization_version: 2
"""

FEATURE_VIEW_CODE = """
# features.py
driver_entity = Entity(name="driver_id", join_keys=["driver_id"])

driver_source = FileSource(
    path="feature_store_data/driver_features.parquet",
    timestamp_field="event_timestamp",
    created_timestamp_column="created",
)

driver_fv = FeatureView(
    name="driver_hourly_stats",
    entities=[driver_entity],
    ttl=timedelta(days=1),
    schema=[
        Field(name="trips_completed", dtype=Int64),
        Field(name="avg_rating", dtype=Float32),
        Field(name="acceptance_rate", dtype=Float32),
        Field(name="conv_rate", dtype=Float32),
    ],
    online=True,
    source=driver_source,
)
"""

print("Feature Store Configuration:")
print(FEATURE_STORE_YAML)
print("Feature View Definition:")
print(FEATURE_VIEW_CODE)

## 3. Point-in-Time Correct Joins

The killer feature of feature stores: when creating training data, join features **as they existed at the time of each training label** — not using future data that wouldn't have been available at prediction time.

In [ ]:
# Demonstrate point-in-time correct joins manually

def point_in_time_join(
    events: pd.DataFrame,
    features: pd.DataFrame,
    entity_col: str,
    timestamp_col: str
) -> pd.DataFrame:
    """
    For each event row, fetch the feature values that were current AT event_timestamp.
    Prevents future data leakage in training datasets.
    """
    result_rows = []
    for _, event in events.iterrows():
        entity_id = event[entity_col]
        event_time = event[timestamp_col]

        # Find the most recent feature row BEFORE the event time
        available = features[
            (features[entity_col] == entity_id) &
            (features["event_timestamp"] <= event_time)
        ].sort_values("event_timestamp").tail(1)

        if not available.empty:
            merged = {**event.to_dict(), **available.iloc[0].drop([entity_col, "event_timestamp", "created"]).to_dict()}
            result_rows.append(merged)

    return pd.DataFrame(result_rows)

# Create sample training events (driver completed a trip → label = tip amount)
training_events = pd.DataFrame({
    "driver_id": np.random.choice(driver_ids[:10], 20),
    "event_timestamp": [datetime(2024, 1, 1) + timedelta(hours=np.random.randint(0, 48)) for _ in range(20)],
    "tip_amount": np.round(np.random.uniform(0, 15, 20), 2)  # label
})

training_data = point_in_time_join(training_events, driver_features, "driver_id", "event_timestamp")
print(f"Training dataset with point-in-time features: {training_data.shape}")
print(training_data[["driver_id", "event_timestamp", "avg_rating", "trips_completed", "tip_amount"]].head(5))

## 4. Online Feature Serving

At inference time, the feature store serves features from the **online store** (low-latency key-value store) — typically sub-10ms retrieval for real-time predictions.

In [ ]:
# Simulate online feature serving with an in-memory store
import time
from typing import Any

class MockOnlineFeatureStore:
    """Simulates low-latency online feature retrieval."""

    def __init__(self, features_df: pd.DataFrame, entity_col: str):
        # Index features by entity key for O(1) lookup
        self._store: dict[int, dict[str, Any]] = {
            row[entity_col]: row.drop(entity_col).to_dict()
            for _, row in features_df.iterrows()
        }

    def get_online_features(self, entity_ids: list[int]) -> pd.DataFrame:
        """Retrieve current feature values for a batch of entity IDs."""
        rows = []
        for eid in entity_ids:
            if eid in self._store:
                rows.append({"driver_id": eid, **self._store[eid]})
        return pd.DataFrame(rows)

# Initialize online store (materialization step)
online_store = MockOnlineFeatureStore(driver_features, "driver_id")

# Benchmark serving latency
request_driver_ids = np.random.choice(driver_ids, 10).tolist()
start = time.perf_counter()
features_for_serving = online_store.get_online_features(request_driver_ids)
latency_ms = (time.perf_counter() - start) * 1000

print(f"Online feature retrieval for {len(request_driver_ids)} drivers: {latency_ms:.2f}ms")
print(features_for_serving[["driver_id", "avg_rating", "trips_completed", "acceptance_rate"]].head(5))

## Exercises

1. **Add a feature transformation**: Create a new derived feature `high_value_driver` (boolean: `trips_completed > 200 AND avg_rating > 4.5`). Add it to the `driver_features` dataframe and update both the offline and online stores. Verify it appears correctly in the training dataset.

2. **Simulate drift monitoring**: Add a second version of `driver_features` with timestamps 7 days later where `avg_rating` has drifted down by 0.3 on average. Implement a function that computes population stability index (PSI) between the two feature snapshots to detect drift.

3. **Install and use real Feast**: Install Feast (`pip install feast`), initialize a real project with `feast init`, adapt the code from this notebook to write actual `entity.py` and `feature_view.py` files, run `feast apply`, materialize features with `feast materialize`, and retrieve them with `store.get_online_features()`.